# 04 — Sentiment Analysis (Fixed Merge + Relative Paths)

This notebook has 2 important features:
1) **Paths** are set up to match the actual project folders (`data/raw`, `data/processed`) and it'll try backup locations if it can't find stuff.
2) **Merging** joins all 6,717 rows from the corpus with the topic data (6,576 rows) using this token-string thing as a bridge, so when we look at *Sentiment by Topic* it actually works properly.

**What you need to have ready:**
- `data/raw/cleaned_corpus.csv`  *(the full text to be analyzed)*
- `data/processed/clean_tokens.parquet`  *(the cleaned up tokens to match topics)*
- `data/processed/docs_agg_map.csv`  *(mapping file with a `text` column that connects everything)*
- `data/processed/doc_topic_matrix_agg.csv`  *(shows which topics belong where)*

**What this creates:**
- `data/processed/sentiment_scored_full.csv`
- `data/processed/sentiment_by_platform.csv`
- `data/processed/sentiment_by_topic.csv`
- `figures/sentiment/*.png`

In [ ]:

import os, re, pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# resolve ROOT: if inside .../notebooks, use parent; else use CWD
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
FIG  = ROOT / "figures" / "sentiment"

# fallbacks for class-demo runs when files are in /mnt/data
FALLBACK = Path("/mnt/data")

PROC.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"RAW:  {RAW}")
print(f"PROC: {PROC}")
print(f"FIG:  {FIG}")


In [ ]:

def find_file(primary: Path, fallback_name: str):
    if primary.exists():
        return primary
    fb = FALLBACK / fallback_name
    if fb.exists():
        return fb
    raise FileNotFoundError(f"Missing file: {primary} (also tried {fb})")

print("Loading all required files...")

p_corpus  = find_file(RAW  / "cleaned_corpus.csv",        "cleaned_corpus.csv")
p_tokens  = find_file(PROC / "clean_tokens.parquet",      "clean_tokens.parquet")
p_map     = find_file(PROC / "docs_agg_map.csv",          "docs_agg_map.csv")
p_weights = find_file(PROC / "doc_topic_matrix_agg.csv",  "doc_topic_matrix_agg.csv")

df_corpus  = pd.read_csv(p_corpus)
df_bridge  = pd.read_parquet(p_tokens)
df_map     = pd.read_csv(p_map)
df_weights = pd.read_csv(p_weights)

print(f"cleaned_corpus.csv rows:        {len(df_corpus)}")
print(f"clean_tokens.parquet rows:      {len(df_bridge)}")
print(f"docs_agg_map.csv rows:          {len(df_map)}")
print(f"doc_topic_matrix_agg.csv rows:  {len(df_weights)}")

# 1) Join corpus (text) with bridge (tokens) by index
cols_keep = [c for c in ["title","link","date_published","text","source","cleaned_text"] if c in df_corpus.columns]
df_full = df_corpus[cols_keep].join(df_bridge[["tokens_nostop"]])

# 2) Build the bridge key used by docs_agg_map, which is a single string of tokens_nostop
df_full["topic_key"] = df_full["tokens_nostop"].apply(' '.join)
# 3) Ccmbine topic map and weights, and compute top topic
df_topics_raw = df_map.join(df_weights)
df_topics_raw["top_topic"] = df_weights.idxmax(axis=1)

# keeps only the first instance of each document key
df_topics = df_topics_raw.drop_duplicates(subset=['text'])
print(f"De-duplicated topic map. Original: {len(df_topics_raw)}, Unique: {len(df_topics)}")

# 4) Final left merge: keep all corpus rows; attach topic rows where available
final_df = pd.merge(
    df_full,
    df_topics.drop(columns=[c for c in ["platform"] if c in df_topics.columns]),
    left_on="topic_key",
    right_on="text",
    how="left",
    suffixes=("_orig","_topic")
)

print(f"\nFinal merged shape: {final_df.shape}")
print("Rows with topic:", final_df["top_topic"].notna().sum())
print("Rows without topic:", final_df["top_topic"].isna().sum())

final_df.head()


In [ ]:

# !pip install transformers torch scipy tqdm -q
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import torch
from tqdm import tqdm

model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def predict_transformers(texts, batch_size=32, max_len=160):
    out=[]
    model.eval()
    for i in tqdm(range(0, len(texts), batch_size), desc="Scoring Sentiment"):
        batch = [str(t) for t in texts[i:i+batch_size]]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=max_len, return_tensors="pt")
        with torch.no_grad():
            logits = model(**enc).logits.numpy()
        out.append(softmax(logits, axis=1))
    return np.vstack(out)

texts_to_score = final_df["cleaned_text"].fillna("").tolist()
probs = predict_transformers(texts_to_score, batch_size=32)

scored_df = final_df.copy()
scored_df[["neg","neu","pos"]] = probs
scored_df["sent_label"] = scored_df[["neg","neu","pos"]].idxmax(axis=1)

out_full = PROC / "sentiment_scored_full.csv"
scored_df.to_csv(out_full, index=False)
print("Saved:", out_full)

scored_df[["source","cleaned_text","neg","neu","pos","sent_label","top_topic"]].head()


In [ ]:

print("Analyzing sentiment by platform...")
by_plat = (scored_df.groupby("source")["sent_label"]
           .value_counts(normalize=True)
           .unstack(fill_value=0)
           .reindex(columns=["neg","neu","pos"], fill_value=0))
by_plat.to_csv(PROC / "sentiment_by_platform.csv")

ax = by_plat.plot(kind="bar", figsize=(8,5), rot=0, title="Sentiment by Platform (YouTube vs Rappler)")
ax.set_ylabel("Proportion")
ax.set_xlabel("Platform")
plt.tight_layout(); plt.savefig(FIG / "sentiment_by_platform.png", dpi=160); plt.close()

overall = scored_df["sent_label"].value_counts(normalize=True).reindex(["neg","neu","pos"]).fillna(0)
overall.plot(kind="bar", figsize=(6,4), rot=0, title="Overall Sentiment")
plt.tight_layout(); plt.savefig(FIG / "overall_sentiment.png", dpi=160); plt.close()

by_plat.round(3)


In [ ]:

print("Analyzing sentiment by topic...")
topic_sent_df = scored_df.dropna(subset=["top_topic"])

tab = (topic_sent_df.groupby("top_topic")["sent_label"]
       .value_counts(normalize=True)
       .unstack(fill_value=0)
       .reindex(columns=["neg","neu","pos"], fill_value=0))

tab.to_csv(PROC / "sentiment_by_topic.csv")

ax = tab.plot(kind="barh", figsize=(10,8), stacked=True, title="Sentiment Distribution per Topic")
ax.set_xlabel("Proportion"); ax.set_ylabel("Topic")
plt.legend(title="Sentiment", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout(); plt.savefig(FIG / "sentiment_by_topic_stacked.png", dpi=160); plt.close()

tab.round(3).head()
